In [10]:
import polars as pl

In [1]:
import os
os.getcwd()

'c:\\Users\\jbogits\\OneDrive - Office of the NYC Comptroller\\Documents - Policy Team!\\Program Areas\\Capital and Infrastructure\\Department of Finance\\Annualized Sales\\Code'

In [ ]:
#os.chdir(r'C:\Users\jbogits\OneDrive - Office of the NYC Comptroller\Documents - Policy Team!\Program Areas\Capital and Infrastructure\Department of Finance\Annualized Sales\Code')

In [3]:
import enrich_analyze as ea

In [13]:
import importlib
importlib.reload(ea)

<module 'enrich_analyze' from 'c:\\Users\\jbogits\\OneDrive - Office of the NYC Comptroller\\Documents - Policy Team!\\Program Areas\\Capital and Infrastructure\\Department of Finance\\Annualized Sales\\Code\\enrich_analyze.py'>

In [5]:
# Step 1: build BBL for archive rows, parse sale dates
sales = ea.step1_add_bbl_and_dates()
print(sales.head())

Step 1: adding BBL to archive rows, parsing sale dates...
  dropped 57,496 completely blank rows (leftover artifacts from the archive CSVs, not real sale records).
  found 1,421 rows with a malformed BBL (not 10 characters) -- blanking these out so they don't cause bad matches later.
  2,099,641 rows. Missing BBL: 1,421. Dates that didn't parse: 0.
  -> data\processed\sales_with_bbl.parquet
shape: (5, 30)
┌─────────┬────────────┬────────────┬────────────┬───┬────────────┬────────────┬──────┬────────────┐
│ borough ┆ neighborho ┆ building_c ┆ tax_class_ ┆ … ┆ bbl        ┆ census_tra ┆ nta  ┆ sale_date_ │
│ ---     ┆ od         ┆ lass_categ ┆ as_of_fina ┆   ┆ ---        ┆ ct_2020    ┆ ---  ┆ parsed     │
│ str     ┆ ---        ┆ ory        ┆ l_roll     ┆   ┆ str        ┆ ---        ┆ str  ┆ ---        │
│         ┆ str        ┆ ---        ┆ ---        ┆   ┆            ┆ str        ┆      ┆ date       │
│         ┆            ┆ str        ┆ str        ┆   ┆            ┆            ┆      

In [6]:
# Flagging properties that sold more than once
sales = ea.step2_flag_repeat_sales()
# skip past the blank-bbl rows sitting at the top, look at real data
print(
    sales.filter(sales["bbl"].is_not_null())
    .select(["bbl", "sale_date_parsed", "sale_sequence"])
    .head(20)
)

non_arms_length_count = sales.filter(sales["non_arms_length_sale"]).height
print(f"non-arm's-length sales: {non_arms_length_count:,} of {len(sales):,}")


Step 2: flagging properties that sold more than once...
  1,308,183 rows are a repeat sale of a property that sold before.
  -> data\processed\sales_with_repeats.parquet
shape: (20, 3)
┌────────────┬──────────────────┬───────────────┐
│ bbl        ┆ sale_date_parsed ┆ sale_sequence │
│ ---        ┆ ---              ┆ ---           │
│ str        ┆ date             ┆ u32           │
╞════════════╪══════════════════╪═══════════════╡
│ 1000010010 ┆ 2003-01-31       ┆ 1             │
│ 1000010010 ┆ 2003-01-31       ┆ 2             │
│ 1000010010 ┆ 2003-01-31       ┆ 3             │
│ 1000010010 ┆ 2010-07-14       ┆ 4             │
│ 1000010010 ┆ 2014-03-01       ┆ 5             │
│ …          ┆ …                ┆ …             │
│ 1000041012 ┆ 2004-07-01       ┆ 1             │
│ 1000041013 ┆ 2004-07-01       ┆ 1             │
│ 1000041014 ┆ 2004-07-01       ┆ 1             │
│ 1000041015 ┆ 2004-07-01       ┆ 1             │
│ 1000041016 ┆ 2004-07-01       ┆ 1             │
└────────────┴─

In [7]:
# Step 3: Joining valuation data
"""This matches on BBL and year, 
so it can only find a match for sales from 2023 onward, 
since that's all the valuation source covers. 
Everything before that will just show blank valuation columns, 
which is expected, not an error."""

sales = ea.step3_join_valuation()

zero_values = sales.filter(sales["val_market_value"] == "0").height
print(f"rows with a $0 valuation: {zero_values:,}")

print(
    sales.filter(sales["val_market_value"].is_not_null())
    .select(["bbl", "sale_date_parsed", "sale_price", "val_market_value", "val_tax_class"])
    .head(10)
)

print(f"rows still showing a literal $0 valuation: {zero_values:,}")

Step 3: joining valuation data (matched by BBL + year)...
  5,750,969 bbl+year combinations had more than one stage recorded -- keeping only the most advanced stage (highest period) for each.
  kept 5,763,392 of 11,556,740 valuation rows after picking one stage per property-year.
  182,936 of 2,099,641 sales matched to a valuation record (only possible for sales from 2023 onward).
  -> data\processed\sales_with_valuation.parquet
rows with a $0 valuation: 0
shape: (10, 5)
┌────────────┬──────────────────┬────────────┬──────────────────┬───────────────┐
│ bbl        ┆ sale_date_parsed ┆ sale_price ┆ val_market_value ┆ val_tax_class │
│ ---        ┆ ---              ┆ ---        ┆ ---              ┆ ---           │
│ str        ┆ date             ┆ str        ┆ str              ┆ str           │
╞════════════╪══════════════════╪════════════╪══════════════════╪═══════════════╡
│ 1000110014 ┆ 2023-06-30       ┆ 2000000    ┆ 12121000         ┆ 2             │
│ 1000110014 ┆ 2024-05-14       

In [14]:
# Step 4: joining PLUTO and filling gaps

sales = ea.step4_join_pluto_and_fill_gaps()

print(
    sales.filter(
        sales["total_units"].is_null() & sales["total_units_filled"].is_not_null()
    ).select(["bbl", "borough", "total_units", "total_units_filled", "pluto_total_units"])
    .head(10)
)

Step 4: joining PLUTO, filling gaps in the sales data...
  filled in total_units for 118,928 rows that were previously missing it.
  2,099,641 rows, 59 columns -> data\processed\sales_enriched.parquet
shape: (10, 5)
┌────────────┬─────────┬─────────────┬────────────────────┬───────────────────┐
│ bbl        ┆ borough ┆ total_units ┆ total_units_filled ┆ pluto_total_units │
│ ---        ┆ ---     ┆ ---         ┆ ---                ┆ ---               │
│ str        ┆ str     ┆ str         ┆ f64                ┆ str               │
╞════════════╪═════════╪═════════════╪════════════════════╪═══════════════════╡
│ 1000110014 ┆ 1       ┆ null        ┆ 20.0               ┆ 20                │
│ 1000110014 ┆ 1       ┆ null        ┆ 20.0               ┆ 20                │
│ 1000110014 ┆ 1       ┆ null        ┆ 20.0               ┆ 20                │
│ 1000110014 ┆ 1       ┆ null        ┆ 20.0               ┆ 20                │
│ 1000110014 ┆ 1       ┆ null        ┆ 20.0               ┆ 20  

In [ ]:
# groups sales into 5-year buckets (2003–2007, 2008–2012, etc.) and shows the no-match count and total for each
sales = sales.with_columns(
    ((sales["sale_date_parsed"].dt.year() // 5) * 5).alias("sale_year_bucket")
)
match_by_era = (
    sales.filter(sales["bbl"].is_not_null())
    .group_by("sale_year_bucket")
    .agg([
        pl.col("pluto_latitude").is_null().sum().alias("no_match"),
        pl.len().alias("total"),
    ])
    .sort("sale_year_bucket")
)
print(match_by_era)

shape: (6, 3)
┌──────────────────┬──────────┬────────┐
│ sale_year_bucket ┆ no_match ┆ total  │
│ ---              ┆ ---      ┆ ---    │
│ i32              ┆ u32      ┆ u32    │
╞══════════════════╪══════════╪════════╡
│ 2000             ┆ 45461    ┆ 245019 │
│ 2005             ┆ 134740   ┆ 512266 │
│ 2010             ┆ 121739   ┆ 405303 │
│ 2015             ┆ 34354    ┆ 435881 │
│ 2020             ┆ 19027    ┆ 415075 │
│ 2025             ┆ 2109     ┆ 84676  │
└──────────────────┴──────────┴────────┘
